Imports + seed

In [1]:
import os
import random

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from scipy.io import loadmat

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision import models

from dataset import BreedDataset

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Load dataset + split test

In [4]:
load_dotenv()
DATASET = os.getenv('DATASET_PATH')
mat_data = loadmat(f'{DATASET}/file_list.mat')

data = []
for num, img_path in enumerate(mat_data['file_list']):
    data.append({
        'img_path': f'{DATASET}/images/{str(img_path[0][0])}',
        'annotation_path': f'{DATASET}/annotation/{mat_data["annotation_list"][num][0][0]}',
        'label': mat_data['labels'][num][0]
    })

df = pd.DataFrame(data)

_, temp_df = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=df['label']
)

_, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['label']
)

Transform + DataLoader

In [5]:
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((480, 480)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

test_dataset = BreedDataset(test_df, transform=val_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Model (EfficientNetV2-M)

In [6]:
def get_model(num_classes=120):
    model = models.efficientnet_v2_m(weights=None)

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, num_classes)
    )

    return model

Evaluation epoch function

In [7]:
@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to('cuda'), labels.to('cuda')

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()

    return total_loss / len(loader), correct / len(loader.dataset)

Load best model

In [8]:
device = 'cuda'

model = get_model(num_classes=120).to(device)

state_dict = torch.load(
    'best_model.pth',
    map_location=device
)

model.load_state_dict(state_dict, strict=True)

<All keys matched successfully>

Run test

In [ ]:
criterion = nn.CrossEntropyLoss()

test_loss, test_acc = eval_epoch(model, test_loader, criterion)

print(f"TEST LOSS: {test_loss:.4f}")
print(f"TEST ACC:  {test_acc:.4f}")